In [ ]:
# Cell 1: load historical log output + coordinated sweep progress.
from pathlib import Path
from collections import deque, Counter
import json, time

# ============================ EDIT THIS PER MACHINE ============================
VM_NAME = "vmA"          # <-- which VM's log to follow (Pod_1 -> VM1, etc.)
# =============================================================================

REPO = Path('/workspace/stable-query-latent')
LOG = Path('/workspace/stable_query_latent_logs') / f'pipeline_{VM_NAME}.log'   # per-VM
OUT_DIR = REPO / 'VICReg_review/heads/cloud_full_sweep_a100'                    # SHARED
EMBED_MANIFEST = REPO / 'game_review_data/embedding_h5.h5.incloud_manifest.json'
TEXT_MANIFEST = REPO / 'game_review_data/build_new_gamedata/text_h5.h5.manifest.json'


def tail(path=LOG, lines=200):
    path = Path(path)
    print(f'log: {path}')
    if not path.exists():
        print('missing log file (nothing written yet for this VM)')
        return
    with path.open('r', encoding='utf-8', errors='replace') as f:
        for line in deque(f, maxlen=lines):
            print(line, end='')


def show_manifest(path):
    path = Path(path)
    print(f'\n=== {path.name} ===')
    if not path.exists():
        print('missing')
        return
    try:
        data = json.loads(path.read_text(encoding='utf-8'))
    except Exception as exc:
        print('bad json:', exc)
        return
    for key in ['status', 'updated_at', 'finished_at', 'error']:
        if key in data:
            print(f'{key}: {data[key]}')


def show_coordination(out_dir=OUT_DIR):
    # Global progress from the SHARED coordination markers (checkpoint/done.json =
    # done; failed.json = failed; status.json = in flight). Every VM's work shows up
    # here since they share this out_dir. (The ledger is now machine-local scratch.)
    out_dir = Path(out_dir)
    print(f'\n=== coordinated sweep @ {out_dir.name} ===')
    if not out_dir.exists():
        print('out_dir not created yet')
        return
    done = failed = inflight = 0
    inflight_by = Counter()
    for d in out_dir.iterdir():
        if not d.is_dir() or d.name == 'VM_parallel':
            continue
        if (d / 'vicreg_review_h5_latest.pt').exists() or (d / 'done.json').exists():
            done += 1
        elif (d / 'failed.json').exists():
            failed += 1
        elif (d / 'status.json').exists():
            inflight += 1
            try:
                inflight_by[json.loads((d / 'status.json').read_text()).get('vm', '?')] += 1
            except Exception:
                pass
    print(f'done={done}  failed={failed}  in-flight={inflight}   in-flight by: {dict(inflight_by)}')
    vmp = out_dir / 'VM_parallel'
    for f in sorted(vmp.glob('*.json')) if vmp.exists() else []:
        try:
            rec = json.loads(f.read_text())
        except Exception:
            rec = {}
        exp = rec.get('expiry')
        live = '' if exp is None else ('(alive)' if exp > time.time() else '(lease expired)')
        print(f"  vm {rec.get('vm', f.stem):16} {live}  {rec.get('info', {})}")


tail(lines=200)
show_manifest(TEXT_MANIFEST)
show_manifest(EMBED_MANIFEST)
show_coordination()
HISTORY_END = LOG.stat().st_size if LOG.exists() else 0

In [ ]:
# Cell 2: start realtime latest log output in the background.
# Re-run this cell to restart the log watcher. It does not stop the training job.
import threading, time
from pathlib import Path
from IPython.display import display
import ipywidgets as widgets


START_OFFSET = globals().get('HISTORY_END', None)
WATCHERS = globals().setdefault('WATCHERS', {})


def stop_watcher(name):
    item = WATCHERS.get(name)
    if item:
        item['stop'].set()
        print(f'stopping {name} watcher')


def read_new_text(path, start_offset):
    if not path.exists():
        return start_offset, ''
    size = path.stat().st_size
    if start_offset is None or start_offset > size:
        start_offset = size
    with path.open('rb') as f:
        f.seek(start_offset)
        data = f.read()
        end_offset = f.tell()
    text = data.decode('utf-8', errors='replace')
    return end_offset, text


def follow(output, stop_event, path=LOG, interval=5, start_offset=START_OFFSET):
    path = Path(path)
    last_offset = start_offset
    last_update = None
    with output:
        print(f'{time.strftime("%Y-%m-%d %H:%M:%S")} | {path}')
        print('-' * 100)
        print('waiting for new log lines after historical output')
    while not stop_event.is_set():
        if path.exists():
            last_offset, new_text = read_new_text(path, last_offset)
            if new_text:
                last_update = time.strftime('%Y-%m-%d %H:%M:%S')
                with output:
                    print('-' * 100)
                    print(f'new output received at {last_update}')
                    print('-' * 100)
                    print(new_text, end='' if new_text.endswith('\n') else '\n')
        else:
            if last_update is None:
                with output:
                    print(f'{time.strftime("%Y-%m-%d %H:%M:%S")} missing log file: {path}')
        stop_event.wait(interval)


stop_watcher('log')
log_output = widgets.Output(layout={'border': '1px solid #ddd', 'height': '520px', 'overflow_y': 'auto'})
display(log_output)
log_stop = threading.Event()
log_thread = threading.Thread(target=follow, args=(log_output, log_stop), daemon=True)
WATCHERS['log'] = {'stop': log_stop, 'thread': log_thread, 'output': log_output}
log_thread.start()
print('log watcher started in background')


In [ ]:
# Cell 3: start realtime system status monitor in the background.
# Re-run this cell to restart the system watcher. It does not stop the training job.
import subprocess, threading, time
from pathlib import Path
from IPython.display import clear_output, display
import ipywidgets as widgets

try:
    import psutil
except ImportError:
    psutil = None
    print('psutil is missing. Run: pip install psutil')

WATCHERS = globals().setdefault('WATCHERS', {})


def stop_watcher(name):
    item = WATCHERS.get(name)
    if item:
        item['stop'].set()
        print(f'stopping {name} watcher')


def _read_int(path):
    try:
        text = Path(path).read_text().strip()
        if text == 'max':
            return None
        return int(text)
    except Exception:
        return None


def get_memory_status():
    limit = _read_int('/sys/fs/cgroup/memory.max')
    used = _read_int('/sys/fs/cgroup/memory.current')
    if limit is None or used is None:
        limit = _read_int('/sys/fs/cgroup/memory/memory.limit_in_bytes')
        used = _read_int('/sys/fs/cgroup/memory/memory.usage_in_bytes')
    if limit and used and limit < 10**18:
        return used / limit * 100, used / 1024**3, limit / 1024**3, 'cgroup'
    if psutil is None:
        return None, None, None, 'unavailable'
    vm = psutil.virtual_memory()
    return vm.percent, vm.used / 1024**3, vm.total / 1024**3, 'host'


def get_gpu_status():
    try:
        out = subprocess.run(
            [
                'nvidia-smi',
                '--query-gpu=index,name,utilization.gpu,memory.used,memory.total,temperature.gpu,power.draw',
                '--format=csv,noheader,nounits',
            ],
            capture_output=True,
            text=True,
            timeout=3,
        ).stdout.strip()
        if not out:
            return ['n/a']
        rows = []
        for line in out.splitlines():
            parts = [part.strip() for part in line.split(',')]
            if len(parts) >= 7:
                rows.append(
                    f'gpu{parts[0]} {parts[1]} | util {float(parts[2]):.0f}% | '
                    f'mem {float(parts[3]) / 1024:.1f}/{float(parts[4]) / 1024:.1f} GiB | '
                    f'temp {float(parts[5]):.0f}C | power {float(parts[6]):.0f}W'
                )
            else:
                rows.append(line)
        return rows
    except Exception as exc:
        return [f'n/a ({exc})']


def system_monitor(output, stop_event, interval=5):
    if psutil is None:
        return
    last_disk = psutil.disk_io_counters()
    last_t = time.time()
    psutil.cpu_percent(interval=None)
    while not stop_event.is_set():
        gpu_rows = get_gpu_status()
        cpu = psutil.cpu_percent(interval=None)
        ram_pct, ram_used, ram_total, ram_source = get_memory_status()
        now_disk = psutil.disk_io_counters()
        now_t = time.time()
        dt = max(now_t - last_t, 1e-6)
        read_mb = (now_disk.read_bytes - last_disk.read_bytes) / 1e6 / dt
        write_mb = (now_disk.write_bytes - last_disk.write_bytes) / 1e6 / dt
        last_disk, last_t = now_disk, now_t

        with output:
            clear_output(wait=True)
            print(time.strftime('%Y-%m-%d %H:%M:%S'))
            print('-' * 100)
            for row in gpu_rows:
                print(f'[gpu] {row}')
            if ram_pct is None:
                ram_msg = 'unavailable'
            else:
                ram_msg = f'{ram_pct:.0f}% ({ram_used:.1f}/{ram_total:.1f} GiB, {ram_source})'
            print(f'[cpu] {cpu:.0f}%')
            print(f'[ram] {ram_msg}')
            print(f'[disk] R {read_mb:.1f} MB/s W {write_mb:.1f} MB/s')
        stop_event.wait(interval)


stop_watcher('system')
system_output = widgets.Output(layout={'border': '1px solid #ddd', 'height': '220px', 'overflow_y': 'auto'})
display(system_output)
system_stop = threading.Event()
system_thread = threading.Thread(target=system_monitor, args=(system_output, system_stop), daemon=True)
WATCHERS['system'] = {'stop': system_stop, 'thread': system_thread, 'output': system_output}
system_thread.start()
print('system watcher started in background')


In [ ]:
# Cell 4: stop background realtime watchers.
WATCHERS = globals().get('WATCHERS', {})
for name, item in list(WATCHERS.items()):
    item['stop'].set()
    print(f'stopped {name} watcher')
